# TP7 : déployer le classifieur CIFAR-10

Objectif : partir du ResNet50 sauvegardé au TP6, l'exporter en TorchScript et en ONNX, comparer la latence de PyTorch et d'ONNX Runtime sur un lot de 100 images, puis le servir comme API REST avec TorchServe : handler, archive `.mar`, serveur, requêtes et description de l'API.

L'énoncé complet, avec les critères de réussite, est dans `enonce.md`.

*Complétez les cellules marquées `# TODO`, dans l'ordre.*

## Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

## Préparation (fournie)

**Données** : CIFAR-10 (Krizhevsky, Nair et Hinton, 2009), fourni par torchvision : des photos couleur de 32 x 32 pixels, en 10 classes (avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion). Le site du jeu ne donne pas de licence ; il demande de citer le rapport *Learning Multiple Layers of Features from Tiny Images* (Krizhevsky, 2009). Tous les TP utilisent les mêmes images : 5000 d'entraînement et 1000 de validation, tirées de l'entraînement de CIFAR-10, et 1000 de test.

Le prétraitement du ResNet agrandit les images en 224 x 224.

In [ ]:
def preparer_archive_cifar10(racine="data"):
    """Place les fichiers de CIFAR-10 dans racine/cifar-10-batches-py sans passer par le serveur d'origine, lent.

    Dans l'ordre : fichiers déjà décompressés ; archive déposée dans la session (cifar-10-python.tar.gz ou
    cifar-10-batches-py.tar.gz, dans le dossier du notebook ou dans racine) ; archive d'origine copiée sur le
    Google Drive de la formation. Si rien n'aboutit, torchvision télécharge ensuite depuis le serveur d'origine ;
    dans tous les cas, il vérifie l'intégrité des fichiers.
    """
    import tarfile
    import urllib.request
    from pathlib import Path

    if Path(racine, "cifar-10-batches-py").exists():
        return
    noms = ("cifar-10-python.tar.gz", "cifar-10-batches-py.tar.gz")
    deposees = [Path(dossier, nom) for dossier in (".", racine) for nom in noms if Path(dossier, nom).exists()]
    if deposees:
        archive = deposees[0]
        print(f"Archive trouvée : {archive}")
    else:
        archive = Path(racine, "cifar-10-python.tar.gz")
        archive.parent.mkdir(parents=True, exist_ok=True)
        url = ("https://drive.usercontent.google.com/download?id=1kyq-1VhOPNr2Bo0w5DiLYuNghOdOWl0a"
               "&export=download&confirm=t")
        print("Téléchargement de CIFAR-10 depuis le Google Drive de la formation (170 Mo)...")
        try:
            urllib.request.urlretrieve(url, archive)
        except OSError as erreur:
            print(f"Google Drive indisponible ({erreur}) : téléchargement depuis le serveur d'origine.")
            return
    try:
        with tarfile.open(archive) as contenu:
            contenu.extractall(racine, filter="data")
    except tarfile.TarError:
        print(f"{archive} n'est pas une archive valide : téléchargement depuis le serveur d'origine.")


def charger_cifar10(dossier="cifar10", n_train=5000, n_val=1000, n_test=1000, seed=0):
    """Images CIFAR-10 au format ImageFolder : dossier/<partie>/<classe>/<image>.png, classes en français.

    Tirages à graine fixe, les mêmes pour tous les TP : n_train et n_val images prises dans les 50 000 images
    d'entraînement de CIFAR-10, sans recouvrement, et n_test dans les 10 000 images de test. Une partie déjà
    présente n'est pas réécrite ; l'écriture passe par un dossier temporaire propre à chaque exécution, renommé à
    la fin : une écriture interrompue n'est jamais prise pour une partie complète, et deux notebooks lancés en même
    temps sur la même machine ne se gênent pas. Les fichiers de CIFAR-10 sont d'abord
    récupérés par preparer_archive_cifar10 (archive déposée dans la session, ou Google Drive de la formation).
    """
    import shutil
    import tempfile
    from pathlib import Path

    import torch
    from torchvision.datasets import CIFAR10

    noms = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
    generateur = torch.Generator().manual_seed(seed)
    ordre = torch.randperm(50000, generator=generateur).tolist()
    parties = {"train": (True, ordre[:n_train]),
               "val": (True, ordre[n_train:n_train + n_val]),
               "test": (False, torch.randperm(10000, generator=generateur)[:n_test].tolist())}
    preparer_archive_cifar10()
    for partie, (entrainement, indices) in parties.items():
        if Path(dossier, partie).exists():
            continue
        source = CIFAR10("data", train=entrainement, download=True)
        Path(dossier).mkdir(parents=True, exist_ok=True)
        temporaire = Path(tempfile.mkdtemp(prefix=f"{partie}.tmp", dir=dossier))
        for numero, indice in enumerate(indices):
            image, label = source[indice]
            sortie = Path(temporaire, noms[label], f"{numero:05d}.png")
            sortie.parent.mkdir(parents=True, exist_ok=True)
            image.save(sortie)
        try:
            temporaire.rename(Path(dossier, partie))
        except OSError:
            # Un autre notebook a fini d'écrire cette partie pendant ce temps : on garde la sienne.
            if not Path(dossier, partie).exists():
                raise
            shutil.rmtree(temporaire)

In [ ]:
charger_cifar10()

TorchServe fonctionne avec Java : la cellule l'installe s'il est absent.

In [ ]:
import shutil

if shutil.which("java") is None:
    !apt-get install -y -qq openjdk-17-jre-headless > /dev/null
!java -version

## Étape 0 : les données et le checkpoint (fournie)

In [ ]:
from pathlib import Path

for partie in ("train", "val", "test"):
    effectifs = {c.name: len(list(c.iterdir())) for c in sorted(Path(f"cifar10/{partie}").iterdir())}
    print(partie, sum(effectifs.values()), effectifs)

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet50_Weights, resnet50
from torchvision.transforms import v2

preprocess = ResNet50_Weights.DEFAULT.transforms()
test_ds = ImageFolder("cifar10/test", transform=preprocess)

Déposez `cifar_best.pt`, sauvegardé au TP6, dans le panneau Fichiers. S'il est absent, la cellule suivante le recrée : 3 époques d'entraînement de la tête d'un ResNet50 gelé, quelques minutes sur GPU.

In [ ]:
def recreer_checkpoint():
    torch.manual_seed(0)
    model = resnet50(weights=ResNet50_Weights.DEFAULT, progress=False)
    for p in model.parameters():
        p.requires_grad_(False)
    model.fc = nn.Linear(2048, 10)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    train_ds = ImageFolder("cifar10/train", transform=preprocess)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    for epoque in range(3):
        for x, y in train_loader:
            loss = nn.CrossEntropyLoss()(model(x.to(device)), y.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    torch.save({"model_state": model.state_dict(), "model_name": "resnet50", "classes": train_ds.classes,
                "epoch": 3}, "cifar_best.pt")

In [ ]:
from pathlib import Path

if not Path("cifar_best.pt").exists():
    recreer_checkpoint()
print(Path("cifar_best.pt").stat().st_size // 2**20, "Mo")

## Étape 1 : recharger le modèle

Rechargez le checkpoint sur CPU (`map_location`), dans un `resnet50()` neuf dont `fc` a 10 sorties, puis passez-le en mode évaluation. Le déploiement se fait sur CPU.

In [ ]:
# TODO : torch.load("cifar_best.pt", map_location="cpu")
checkpoint = ...
model = resnet50()
model.fc = nn.Linear(2048, 10)
# TODO : model.load_state_dict(...) avec checkpoint["model_state"], puis model.eval()
...
CLASSES = checkpoint["classes"]
print({cle: valeur for cle, valeur in checkpoint.items() if cle != "model_state"})

Le lot de 100 images de test, 10 par classe, et les prédictions de référence.

In [ ]:
lot = torch.stack([test_ds[i][0] for i in range(0, 1000, 10)])
etiquettes = torch.tensor([test_ds.targets[i] for i in range(0, 1000, 10)])
with torch.no_grad():
    logits_torch = model(lot)
print(lot.shape, "accuracy sur le lot :", round((logits_torch.argmax(1) == etiquettes).float().mean().item(), 3))

## Étape 2 : TorchScript

Tracez le modèle avec une entrée d'exemple de forme (1, 3, 224, 224), sauvegardez-le dans `cifar_traced.pt`, rechargez-le et vérifiez qu'il donne les mêmes sorties.

In [ ]:
# TODO : torch.jit.trace, save("cifar_traced.pt"), torch.jit.load
traced = ...
charge = ...
with torch.no_grad():
    ecart = (charge(lot) - logits_torch).abs().max().item()
print("écart max :", ecart)
assert ecart < 1e-3

## Étape 3 : ONNX

Exportez le modèle dans `cifar.onnx`, avec une entrée nommée `image`, une sortie nommée `logits`, une dimension de batch variable (`dynamic_shapes`), et les poids dans le même fichier (`external_data=False`).

In [ ]:
# TODO : torch.onnx.export vers "cifar.onnx", input_names=["image"], output_names=["logits"],
#        dynamic_shapes=({0: "batch"},), external_data=False, verbose=False
print(Path("cifar.onnx").stat().st_size // 2**20, "Mo")

Exécutez-le avec ONNX Runtime sur le lot de 100 images et comparez aux sorties de PyTorch.

In [ ]:
import onnxruntime as ort

# TODO : InferenceSession sur CPU, puis session.run sur le lot (tableau NumPy)
session = ...
logits_onnx = ...
ecart = abs(logits_onnx - logits_torch.numpy()).max()
print("écart max :", ecart)
assert ecart < 1e-2 and (logits_onnx.argmax(1) == logits_torch.argmax(1).numpy()).all()

## Étape 4 : la latence sur CPU (fournie)

Chaque moteur traite le lot de 100 images : une exécution de chauffe, puis la moyenne de trois.

In [ ]:
import time


def latence(executer, repetitions=3):
    executer()
    debut = time.perf_counter()
    for _ in range(repetitions):
        executer()
    return (time.perf_counter() - debut) / repetitions

In [ ]:
with torch.no_grad():
    moteurs = {
        "PyTorch": lambda: model(lot),
        "TorchScript": lambda: charge(lot),
        "ONNX Runtime": lambda: session.run(None, {"image": lot.numpy()}),
    }
    for nom, executer in moteurs.items():
        print(f"{nom:13s} {latence(executer):.2f} s pour 100 images")

Question : pour un service qui reçoit une image à la fois, quelle mesure faudrait-il faire en plus ?

*Votre réponse :*

## Étape 5 : le handler TorchServe

Le handler reçoit les images envoyées par HTTP. Complétez :

- `PREPARER` : redimensionnement à 232, recadrage central à 224, tenseur flottant entre 0 et 1, normalisation ImageNet (moyenne `[0.485, 0.456, 0.406]`, écart-type `[0.229, 0.224, 0.225]`) : le même prétraitement qu'à l'entraînement ;
- `postprocess` : softmax, puis pour chaque image la classe la plus probable et sa probabilité.

In [ ]:
%%writefile cifar_handler.py
import io

import torch
from PIL import Image
from torchvision.transforms import v2
from ts.torch_handler.base_handler import BaseHandler

CLASSES = ["automobile", "avion", "bateau", "camion", "cerf", "chat", "cheval", "chien", "grenouille", "oiseau"]
PREPARER = v2.Compose([
    # TODO : v2.Resize(232) puis v2.CenterCrop(224)
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    # TODO : v2.Normalize avec la moyenne et l'écart-type ImageNet
])


class CifarHandler(BaseHandler):
    def preprocess(self, data):
        images = [Image.open(io.BytesIO(ligne.get("data") or ligne.get("body"))).convert("RGB") for ligne in data]
        return torch.stack([PREPARER(image) for image in images])

    def postprocess(self, output):
        # TODO : probas = softmax de output sur la dimension 1
        probas = ...
        return [{"classe": CLASSES[int(p.argmax())], "probabilite": round(float(p.max()), 3)} for p in probas]

## Étape 6 : l'archive `.mar`

Empaquetez `cifar_traced.pt` et le handler dans `model_store`, sous le nom `cifar`.

In [ ]:
!mkdir -p model_store
# TODO : torch-model-archiver --model-name cifar --version 1.0 --serialized-file ... --handler ...
#        --export-path model_store --force
!ls -lh model_store

## Étape 7 : démarrer le serveur (fournie)

`--disable-token-auth` désactive le jeton d'authentification : à réserver à un environnement de test. `config.properties` fixe les ports (8080, le port par défaut, est déjà pris dans Colab) et un worker par modèle.

In [ ]:
%%writefile config.properties
inference_address=http://127.0.0.1:7080
management_address=http://127.0.0.1:7081
metrics_address=http://127.0.0.1:7082
default_workers_per_model=1

In [ ]:
import time

import requests


def attendre_serveur(url="http://127.0.0.1:7080/ping", essais=60):
    for _ in range(essais):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return requests.get(url).json()
        except requests.ConnectionError:
            pass
        time.sleep(2)
    raise RuntimeError("TorchServe ne répond pas : voir torchserve.log")

In [ ]:
# < /dev/null : sans cela, la cellule attend l'arrêt du serveur
!torchserve --start --ncs --ts-config config.properties --model-store model_store --models cifar=cifar.mar --disable-token-auth < /dev/null > torchserve.log 2>&1
attendre_serveur()

## Étape 8 : interroger l'API

Envoyez une image d'avion et une image de chat avec `curl` sur `/predictions/cifar`.

In [ ]:
image_avion = sorted(Path("cifar10/test/avion").iterdir())[0]
image_chat = sorted(Path("cifar10/test/chat").iterdir())[0]
print(image_avion, image_chat)

In [ ]:
# TODO : deux requêtes curl -s -X POST sur http://127.0.0.1:7080/predictions/cifar avec -T {chemin}

## Étape 9 : la description de l'API

Une requête OPTIONS sur la racine renvoie la description OpenAPI des points d'entrée : c'est le contrat d'interface que consommera tout client du service. Enregistrez-la dans `api_cifar.json` et affichez la route de prédiction.

In [ ]:
import json

description = requests.options("http://127.0.0.1:7080/").json()
Path("api_cifar.json").write_text(json.dumps(description, indent=2))
print(description["openapi"], list(description["paths"]))

In [ ]:
route = description["paths"]["/predictions/{model_name}"]["post"]
print(route["description"])
print([parametre["name"] for parametre in route["parameters"]])

## Bonus

**B1. L'accuracy à travers l'API.** Envoyez 10 images de test de chaque classe avec `requests` et calculez l'accuracy des réponses.

In [ ]:
# TODO : 100 requêtes, puis l'accuracy

## Arrêter le serveur (fournie)

In [ ]:
!torchserve --stop